# LightSVTR

> EncoderWithLightSVTR, the PP-OCRv6 medium recognition neck.


In [ ]:
#| default_exp layers.svtr


#| export
The exported graph is the inference network. Dropout is gone. Each Conv-BN-Swish stays unfused, and each LayerNorm is a learned scale and bias. Two global transformer blocks sit between a 1×7 depthwise conv and an additive 1×1 skip. The NRTR decoder is not in the checkpoint.


In [ ]:
#| hide
from nbdev import show_doc


In [ ]:
#| export
#| notest
from __future__ import annotations

import numpy as np
from max.graph import TensorValue, Weight, ops
from max.nn.layer import Module
from max.nn.linear import Linear

from solid_lens.config import DEVICE, DTYPE
from solid_lens.layers.conv import conv2d
from solid_lens.layers.lcnet import BatchNorm


In [ ]:
#| export
#| notest
class LayerNorm(Module):
    def __init__(self, channels: int, eps: float) -> None:
        super().__init__()
        self.eps = eps
        self.weight = Weight("weight", DTYPE, (channels,), DEVICE)
        self.bias = Weight("bias", DTYPE, (channels,), DEVICE)

    def __call__(self, x: TensorValue) -> TensorValue:
        return ops.layer_norm(x, self.weight, self.bias, epsilon=self.eps)


The checkpoint does not contain a LayerNorm op. It contains the scale and the bias, and `weights` finds those as a Mul followed by an Add. This module is the pair those tensors belong to.


In [ ]:
#| export
#| notest
class ConvBNSwish(Module):
    "1x1 projection used by the skip and the channel reduction."

    def __init__(self, in_channels: int, out_channels: int, eps: float) -> None:
        super().__init__()
        self.conv = conv2d(in_channels, out_channels, 1, bias=False)
        self.bn = BatchNorm(out_channels, eps)

    def __call__(self, x: TensorValue) -> TensorValue:
        return ops.silu(self.bn(self.conv(x)))


In [ ]:
#| export
#| notest
class LocalMixer(Module):
    "1xk depthwise conv along the text width, added back onto its input."

    def __init__(self, channels: int, kernel: int, eps: float) -> None:
        super().__init__()
        pad = kernel // 2
        self.conv = conv2d(
            channels,
            channels,
            (1, kernel),
            padding=(0, 0, pad, pad),
            groups=channels,
            bias=False,
        )
        self.bn = BatchNorm(channels, eps)

    def __call__(self, x: TensorValue) -> TensorValue:
        return x + ops.silu(self.bn(self.conv(x)))


The local mixer is a 1×7 depthwise conv along the sequence, then batch-norm and SiLU, added back to its input. It does not mix height. By the time it runs, the map is one row tall.


## Attention


In [ ]:
#| export
#| notest
class SVTRBlock(Module):
    "Pre-norm global attention and a Swish MLP. Dropout is identity here."

    def __init__(self, dim: int, num_heads: int, mlp_ratio: float, eps: float) -> None:
        super().__init__()
        if dim % num_heads != 0:
            raise ValueError(f"{dim} channels is not divisible by {num_heads} heads")
        self.num_heads = num_heads
        self.head_dim = dim // num_heads
        self.norm1 = LayerNorm(dim, eps)
        self.qkv = Linear(dim, dim * 3, DTYPE, DEVICE, has_bias=True)
        self.proj = Linear(dim, dim, DTYPE, DEVICE, has_bias=True)
        self.norm2 = LayerNorm(dim, eps)
        hidden = int(dim * mlp_ratio)
        self.fc1 = Linear(dim, hidden, DTYPE, DEVICE, has_bias=True)
        self.fc2 = Linear(hidden, dim, DTYPE, DEVICE, has_bias=True)

    def __call__(self, x: TensorValue) -> TensorValue:
        x = x + self._attend(self.norm1(x))
        return x + self.fc2(ops.silu(self.fc1(self.norm2(x))))

    def _attend(self, x: TensorValue) -> TensorValue:
        # x is (batch, width, channels). The scale is 1/sqrt(head_dim) and
        # multiplies q only, matching the exported graph.
        batch, width, _ = x.shape[0], x.shape[1], x.shape[2]
        heads = self.num_heads
        head_dim = self.head_dim
        mixed = ops.permute(
            ops.reshape(self.qkv(x), [batch, width, 3, heads, head_dim]),
            [2, 0, 3, 1, 4],
        )
        query, key, value = ops.split(mixed, [1, 1, 1], axis=0)
        scale = ops.constant(np.float32(head_dim**-0.5), DTYPE, device=DEVICE)
        query = ops.squeeze(query, axis=0) * scale
        key = ops.squeeze(key, axis=0)
        value = ops.squeeze(value, axis=0)
        scores = ops.softmax(
            ops.matmul(query, ops.permute(key, [0, 1, 3, 2])),
            axis=-1,
        )
        context = ops.matmul(scores, value)
        context = ops.permute(context, [0, 2, 1, 3])
        context = ops.reshape(context, [batch, width, heads * head_dim])
        return self.proj(context)


The scale multiplies the query only. That is how the exported graph is written, and it is not the same op as scaling the scores after the matmul when the dtype is not exact. The MLP is SiLU, not GELU. Dropout is absent, so it is not in the module.


In [ ]:
#| export
#| notest
class LightSVTR(Module):
    "Local conv, global blocks, then an additive skip of the backbone features."

    def __init__(
        self,
        in_channels: int,
        dim: int,
        depth: int,
        num_heads: int,
        mlp_ratio: float,
        local_kernel: int,
        eps: float,
        final_eps: float,
    ) -> None:
        super().__init__()
        # Skip is defined first because the exporter emits it first.
        self.skip = ConvBNSwish(in_channels, dim, eps)
        self.reduce = ConvBNSwish(in_channels, dim, eps)
        self.local = LocalMixer(dim, local_kernel, eps)
        self.depth = depth
        for index in range(depth):
            setattr(self, f"block{index}", SVTRBlock(dim, num_heads, mlp_ratio, eps))
        self.norm = LayerNorm(dim, final_eps)

    def __call__(self, x: TensorValue) -> TensorValue:
        # x is NCHW with height 1. Attention runs on (batch, width, channels).
        skip = _as_sequence(self.skip(x))
        hidden = self.local(self.reduce(x))
        sequence = _as_sequence(hidden)
        for index in range(self.depth):
            sequence = getattr(self, f"block{index}")(sequence)
        return self.norm(sequence) + skip


The skip is constructed first because the exporter emits it first. `final_eps` is 1e-6 and applies only to the norm after the last block. The blocks themselves use `eps`, which is 1e-5.


In [ ]:
#| export
#| notest
def _as_sequence(x: TensorValue) -> TensorValue:
    return ops.permute(ops.squeeze(x, axis=2), [0, 2, 1])
